# Mental Health Signal Detector — Fine-tuning DistilBERT
**Artefact School of Data — Bootcamp Data Science, Mars 2026**

⚡ Assure-toi d'avoir activé le GPU : `Exécution > Modifier le type d'exécution > GPU (T4)`

In [ ]:
# Étape 1 : install des dépendances
# ⚠️ IMPORTANT : après cette cellule, clique sur 'Redémarrer la session' quand Colab le propose
!pip install -q -U transformers accelerate datasets scikit-learn

In [ ]:
# Étape 2 : vérification GPU (lancer après redémarrage)
import torch
print('GPU disponible :', torch.cuda.is_available())
print('Device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
import transformers
print('Transformers version :', transformers.__version__)

In [ ]:
# Étape 3 : monter Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/mh_detector/fine_tuned'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Modèle sauvegardé dans :', OUTPUT_DIR)

In [ ]:
# Étape 4 : chargement et préparation des données
import re
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset

def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

ds = load_dataset('dair-ai/emotion')
train_df = ds['train'].to_pandas()
test_df  = ds['test'].to_pandas()

# Binarisation : sadness(0) + fear(4) = détresse
DISTRESS = {0, 4}
for df in [train_df, test_df]:
    df['label'] = df['label'].apply(lambda x: 1 if x in DISTRESS else 0)
    df['text']  = df['text'].apply(clean_text)

print('Train:', len(train_df), '| distribution:', train_df['label'].value_counts().to_dict())
print('Test: ', len(test_df),  '| distribution:', test_df['label'].value_counts().to_dict())

In [ ]:
# Étape 5 : fine-tuning DistilBERT
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['text','label']]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text','label']]).map(tokenize, batched=True)
train_ds = train_ds.rename_column('label', 'labels')
test_ds  = test_ds.rename_column('label', 'labels')
train_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
test_ds.set_format('torch',  columns=['input_ids','attention_mask','labels'])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted'),
    }

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Étape 6 : sauvegarde
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('✅ Modèle sauvegardé dans Google Drive :', OUTPUT_DIR)

In [ ]:
# Étape 7 : évaluation finale
results = trainer.evaluate()
print('\n=== Résultats DistilBERT ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Étape 8 : télécharger le modèle en zip
import shutil
shutil.make_archive('/content/fine_tuned', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/fine_tuned.zip')
print('⬇️  Dézippe dans mental-health-signal-detector/models/fine_tuned/')